# 01 — Data Collection

Fetch 1-minute candlestick data for all markets that passed the liquidity filter,
plus crypto spot data, Deribit DVOL, and LunarCrush sentiment.

**Prerequisites**: Run `00_market_selection.ipynb` first.

**Outputs**:
- `data/raw/kalshi/{ticker}.parquet` — one file per prediction market
- `data/raw/crypto/crypto_1m_{symbol}.parquet` — BTC/ETH/SOL 1-minute bars
- `data/raw/deribit/dvol_btc.parquet`, `dvol_eth.parquet`
- `data/raw/lunarcrush/sentiment_btc.parquet`
- `data/processed/prob_mid_aligned.parquet` — all prediction markets on a common 1m UTC index

In [1]:
import sys
sys.path.append('..')

import os
import yaml
import time
import pandas as pd
import numpy as np
from pathlib import Path
from dotenv import load_dotenv
from tqdm.notebook import tqdm
from src.fetch import (
    fetch_kalshi_candlesticks, save_kalshi_market, load_kalshi_market,
    fetch_deribit_dvol, save_deribit_dvol,
    fetch_lunarcrush_sentiment, save_lunarcrush,
    fetch_binance_klines, save_crypto,
)

# Load API keys from .env in the project root
load_dotenv('../.env')

with open('../config.yaml') as f:
    cfg = yaml.safe_load(f)

API_KEY = os.environ.get('KALSHI_API_KEY', None)
LUNARCRUSH_KEY = os.environ.get('LUNARCRUSH_API_KEY', None)

# Date range for data collection
START_DATE = '2022-01-01'
END_DATE   = '2026-04-01'

start_ts = int(pd.Timestamp(START_DATE, tz='UTC').timestamp())
end_ts   = int(pd.Timestamp(END_DATE,   tz='UTC').timestamp())

# Load the final market list from notebook 00
final_markets = pd.read_parquet('../data/raw/kalshi/final_market_list.parquet')
print(f'Markets to fetch: {len(final_markets)}')
print(final_markets[['ticker', 'tier']].to_string(index=False))

Markets to fetch: 53
                          ticker  tier
    KXBTCMAXY-25-DEC31-129999.99     1
    KXBTCMAXY-25-DEC31-139999.99     1
KXBTCMAX150-25-26MAY31-149999.99     1
KXBTCMAX150-25-26MAR31-149999.99     1
KXBTCMAX150-25-26APR30-149999.99     1
KXBTCMAX150-25-26FEB28-149999.99     1
              KXBTCMAX100-26-MAR     1
KXBTCMAX150-25-26JAN31-149999.99     1
              KXBTCMAX100-26-FEB     1
              KXBTCMAX100-26-APR     1
    KXBTCMAXY-25-DEC31-149999.99     1
  KXBTCMAX150-25-DEC31-149999.99     1
             KXBTCMAX100-26-JUNE     1
      KXBTCMAXY-26DEC31-99999.99     1
              KXBTCMAX100-26-MAY     1
    KXBTCMAXY-25-DEC31-199999.99     1
    KXBTCMAXY-25-DEC31-169999.99     1
    KXBTCMAXY-25-DEC31-159999.99     1
    KXBTCMAXY-25-DEC31-189999.99     1
              KXBTCMAX100-26-DEC     1
              KXBTCMAX100-26-SEP     1
     KXBTCMAXY-26DEC31-119999.99     1
     KXBTCMAXY-26DEC31-109999.99     1
    KXBTCMAXY-25-DEC31-179999.99     1
    

/Users/kianjagtiani/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


## 1. Fetch Kalshi 1-Minute Candlesticks

One parquet file per market. Historical (settled) markets use the `/historical/` endpoint.

In [2]:
kalshi_dir = Path('../data/raw/kalshi')
all_markets_catalog = pd.read_parquet(kalshi_dir / 'market_catalog.parquet')

failed = []

for _, row in tqdm(final_markets.iterrows(), total=len(final_markets), desc='Kalshi markets'):
    ticker = row['ticker']
    out_path = kalshi_dir / f'{ticker}.parquet'

    if out_path.exists():
        print(f'  {ticker}: already exists, skipping')
        continue

    # Determine if this is a settled market
    market_info = all_markets_catalog[all_markets_catalog['ticker'] == ticker]
    status = str(market_info['status'].values[0]).lower() if not market_info.empty else ''
    is_hist = status in ('settled', 'closed')

    try:
        df = fetch_kalshi_candlesticks(
            ticker=ticker,
            start_ts=start_ts,
            end_ts=end_ts,
            period_interval=1,   # 1-minute bars
            is_historical=is_hist,
            api_key=API_KEY,
        )
        time.sleep(0.05)

        if df.empty:
            print(f'  {ticker}: no data returned')
            failed.append(ticker)
            continue

        save_kalshi_market(ticker, df)
        print(f'  {ticker}: {len(df)} bars, saved')

    except Exception as e:
        print(f'  {ticker}: ERROR — {e}')
        failed.append(ticker)

print(f'\nFailed: {failed}')

Kalshi markets:   0%|          | 0/53 [00:00<?, ?it/s]

  KXBTCMAXY-25-DEC31-129999.99: 134546 bars, saved
  KXBTCMAXY-25-DEC31-139999.99: 105103 bars, saved
  KXBTCMAX150-25-26MAY31-149999.99: 102525 bars, saved
  KXBTCMAX150-25-26MAR31-149999.99: 76668 bars, saved
  KXBTCMAX150-25-26APR30-149999.99: 97786 bars, saved
  KXBTCMAX150-25-26FEB28-149999.99: 94089 bars, saved
  KXBTCMAX100-26-MAR: 44426 bars, saved
  KXBTCMAX150-25-26JAN31-149999.99: 75031 bars, saved
  KXBTCMAX100-26-FEB: 27626 bars, saved
  KXBTCMAX100-26-APR: 49739 bars, saved
  KXBTCMAXY-25-DEC31-149999.99: 143941 bars, saved
  KXBTCMAX150-25-DEC31-149999.99: 93413 bars, saved
  KXBTCMAX100-26-JUNE: 59528 bars, saved
  KXBTCMAXY-26DEC31-99999.99: 55194 bars, saved
  KXBTCMAX100-26-MAY: 54354 bars, saved
  KXBTCMAXY-25-DEC31-199999.99: 81123 bars, saved
  KXBTCMAXY-25-DEC31-169999.99: 47031 bars, saved
  KXBTCMAXY-25-DEC31-159999.99: 69937 bars, saved
  KXBTCMAXY-25-DEC31-189999.99: 36151 bars, saved
  KXBTCMAX100-26-DEC: 31546 bars, saved
  KXBTCMAX100-26-SEP: 22921 bars, s

## 2. Crypto Spot Data — Binance

Option A (recommended for large date ranges): use `binance-historical-data` bulk downloader.
Option B (smaller ranges): direct Binance REST API via `fetch_binance_klines`.

In [9]:
import requests
import zipfile
import io
from datetime import date
from dateutil.relativedelta import relativedelta

crypto_dir = Path('../data/raw/crypto')
crypto_dir.mkdir(parents=True, exist_ok=True)

symbols = cfg['crypto_assets']  # ['BTCUSDT', 'ETHUSDT', 'SOLUSDT']

BASE_URL = "https://data.binance.vision/data/spot/monthly/klines"

def download_month(symbol: str, year: int, month: int, out_dir: Path) -> bool:
    filename = f"{symbol}-1m-{year}-{month:02d}.zip"
    url = f"{BASE_URL}/{symbol}/1m/{filename}"
    try:
        r = requests.get(url, timeout=60)
        if r.status_code == 404:
            return False  # month not available yet
        r.raise_for_status()
        with zipfile.ZipFile(io.BytesIO(r.content)) as z:
            z.extractall(out_dir)
        return True
    except Exception as e:
        print(f"    {filename}: {e}")
        return False

start = date(2022, 1, 1)
end   = date(2026, 4, 1)

for symbol in symbols:
    sym_dir = crypto_dir / symbol
    sym_dir.mkdir(parents=True, exist_ok=True)
    print(f"\nDownloading {symbol}...")
    
    cur = start.replace(day=1)
    downloaded = 0
    while cur < end:
        csv_name = f"{symbol}-1m-{cur.year}-{cur.month:02d}.csv"
        if (sym_dir / csv_name).exists():
            cur += relativedelta(months=1)
            downloaded += 1
            continue
        ok = download_month(symbol, cur.year, cur.month, sym_dir)
        if ok:
            downloaded += 1
        cur += relativedelta(months=1)
    
    print(f"  {symbol}: {downloaded} monthly files ready")

print('\nBulk download complete.')


  BTCUSDT: 51 monthly files ready

  ETHUSDT: 51 monthly files ready

  SOLUSDT: 51 monthly files ready

Bulk download complete.


In [17]:
# Load, combine, and save as parquet for fast access
import importlib, src.fetch as _f; importlib.reload(_f)                                                                 
from src.fetch import load_binance_bulk, save_crypto   

for symbol in symbols:
    sym_dir = crypto_dir / symbol
    print(f'Processing {symbol}...')
    try:
        df = load_binance_bulk(symbol, data_dir=sym_dir)
        save_crypto(symbol, df)
        print(f'  {symbol}: {len(df):,} bars ({df.timestamp_utc.min().date()} to {df.timestamp_utc.max().date()})')
    except FileNotFoundError as e:
        print(f'  {symbol}: {e}')

Processing BTCUSDT...


Loading BTCUSDT: 100%|████████████████████████████████████████████████████████████████████| 51/51 [00:01<00:00, 27.23it/s]


  BTCUSDT: 2,233,360 bars (2022-01-01 to 2026-03-31)
Processing ETHUSDT...


Loading ETHUSDT: 100%|████████████████████████████████████████████████████████████████████| 51/51 [00:01<00:00, 28.83it/s]


  ETHUSDT: 2,233,360 bars (2022-01-01 to 2026-03-31)
Processing SOLUSDT...


Loading SOLUSDT: 100%|████████████████████████████████████████████████████████████████████| 51/51 [00:01<00:00, 29.68it/s]


  SOLUSDT: 2,233,359 bars (2022-01-01 to 2026-03-31)


## 3. Deribit DVOL (Free)

In [18]:
for currency in ['BTC', 'ETH']:
    print(f'Fetching DVOL for {currency}...')
    try:
        dvol = fetch_deribit_dvol(currency)
        save_deribit_dvol(currency, dvol)
        print(f'  {len(dvol)} daily records ({dvol.timestamp_utc.min().date()} to {dvol.timestamp_utc.max().date()})')
    except Exception as e:
        print(f'  ERROR: {e}')

Fetching DVOL for BTC...
  384 daily records (2026-03-23 to 2026-04-08)
Fetching DVOL for ETH...
  384 daily records (2026-03-23 to 2026-04-08)


## 4. LunarCrush Sentiment (Free Tier)

In [14]:
for coin in ['btc', 'eth']:
    print(f'Fetching sentiment for {coin}...')
    try:
        sent = fetch_lunarcrush_sentiment(coin=coin, api_key=LUNARCRUSH_KEY)
        if not sent.empty:
            save_lunarcrush(coin, sent)
            print(f'  {len(sent)} hourly records')
        else:
            print(f'  No data returned (may need API key)')
    except Exception as e:
        print(f'  ERROR: {e}')

Fetching sentiment for btc...
  ERROR: 401 Client Error: Unauthorized for url: https://lunarcrush.com/api4/public/coins/btc/v1
Fetching sentiment for eth...
  ERROR: 401 Client Error: Unauthorized for url: https://lunarcrush.com/api4/public/coins/eth/v1


## 5. Build Aligned Probability Mid-Price Table

Combine all Kalshi markets onto a common 1-minute UTC DatetimeIndex.
Each column is one market's `prob_mid`.

In [21]:
all_probs = {}                                                                                                          
                                                        
for ticker in final_markets['ticker']:
  path = kalshi_dir / f'{ticker}.parquet'
  if not path.exists():                                                                                               
      continue
  df = pd.read_parquet(path)                                                                                          
  if 'prob_mid' not in df.columns or 'timestamp_utc' not in df.columns:
      continue                                                                                                        
  df = df.set_index('timestamp_utc').sort_index()
  df = df[~df.index.duplicated(keep='first')]                                                                         
  all_probs[ticker] = df['prob_mid']                    
                                                                                                                      
if all_probs:
  prob_df = pd.DataFrame(all_probs)                                                                                   
  proc_dir = Path('../data/processed')                  
  proc_dir.mkdir(parents=True, exist_ok=True)
  prob_df.to_parquet(proc_dir / 'prob_mid_aligned.parquet')
  print(f'Saved prob_mid_aligned: {prob_df.shape} (rows x markets)')                                                  
  print(f'Date range: {prob_df.index.min()} to {prob_df.index.max()}')                                                
  print(f'\nData coverage per market:')                                                                               
  print((prob_df.notna().sum() / len(prob_df) * 100).round(1).to_string())                                            
else:                                                                                                                   
  print('No market data found — check that notebook 00 and the fetch step completed.') 

Saved prob_mid_aligned: (416225, 53) (rows x markets)
Date range: 2024-11-11 23:32:00+00:00 to 2026-04-01 00:00:00+00:00

Data coverage per market:
KXBTCMAXY-25-DEC31-129999.99        32.3
KXBTCMAXY-25-DEC31-139999.99        25.2
KXBTCMAX150-25-26MAY31-149999.99    24.6
KXBTCMAX150-25-26MAR31-149999.99    18.4
KXBTCMAX150-25-26APR30-149999.99    23.5
KXBTCMAX150-25-26FEB28-149999.99    22.6
KXBTCMAX100-26-MAR                  10.7
KXBTCMAX150-25-26JAN31-149999.99    18.0
KXBTCMAX100-26-FEB                   6.6
KXBTCMAX100-26-APR                  11.9
KXBTCMAXY-25-DEC31-149999.99        34.6
KXBTCMAX150-25-DEC31-149999.99      22.4
KXBTCMAX100-26-JUNE                 14.3
KXBTCMAXY-26DEC31-99999.99          13.3
KXBTCMAX100-26-MAY                  13.1
KXBTCMAXY-25-DEC31-199999.99        19.5
KXBTCMAXY-25-DEC31-169999.99        11.3
KXBTCMAXY-25-DEC31-159999.99        16.8
KXBTCMAXY-25-DEC31-189999.99         8.7
KXBTCMAX100-26-DEC                   7.6
KXBTCMAX100-26-SEP              